In [1]:
# Importamos las librerías básicas

import numpy as np
import pandas as pd

# modificamos la configuración para ver todas las columnas al mostrar dataframes
pd.set_option("display.max_columns", None)


In [2]:
# Carga de datasets

# Cargar dataset CSV (campañas de marketing)
bank_df = pd.read_csv( "../Data/DataRaw/bank-additional.csv", sep=",", index_col=0)  # usamos coma como separador

# Cargar dataset Excel (detalles de clientes con varias hojas)
customer_xlsx = pd.ExcelFile("../Data/DataRaw/customer-details.xlsx")
customer_2012 = pd.read_excel(customer_xlsx, sheet_name="2012", index_col=0)
customer_2013 = pd.read_excel(customer_xlsx, sheet_name="2013", index_col=0)
customer_2014 = pd.read_excel(customer_xlsx, sheet_name="2014", index_col=0)

# Utilizamos index_col=0 para indicar que la primera columna se use como índice

In [3]:
bank_df.sample(5)

,age,job,marital,education,default,housing,loan,contact,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,date,latitude,longitude,id_
37982,52.0,technician,MARRIED,professional.course,0.0,1.0,1.0,cellular,211,1,999,1,FAILURE,-3.4,"92,379","-29,8","0,797","5017,5",yes,8-marzo-2015,37.574,-114.729,3ad74ab4-9e49-432a-8056-b47fbdd9deb9
8996,31.0,admin.,SINGLE,basic.9y,0.0,0.0,0.0,telephone,105,2,999,0,NONEXISTENT,1.4,"94,465","-41,8","4,866","5228,1",no,13-enero-2018,39.978,-83.434,6b1e10d0-d441-4440-839a-c2c628526280
8417,40.0,blue-collar,DIVORCED,basic.9y,NaN,0.0,1.0,telephone,1259,6,999,0,NONEXISTENT,1.4,"94,465","-41,8","4,864","5228,1",yes,23-marzo-2016,37.394,-111.069,1078494d-65f3-434e-aa62-41b7b54a2b21
40992,48.0,technician,MARRIED,university.degree,0.0,1.0,1.0,cellular,337,2,999,0,NONEXISTENT,-1.1,"94,601","-49,5","1,008","4963,6",yes,19-marzo-2018,48.917,-101.659,e37bc444-7919-4798-babf-9a5f8eda9f65
26015,42.0,technician,MARRIED,basic.9y,0.0,0.0,0.0,cellular,294,2,999,0,NONEXISTENT,-0.1,"93,2",-42,"4,12","5195,8",no,28-octubre-2016,46.868,-117.890,08e88044-1319-4168-a9c4-c4445b2ef152


Antes de realizar las transformaciones vamos a hacer una copia de nuestro set de datos para trabajar con ella. La primera transformación va a ser pasar la columna age de float a int ya que los valores son edades de los clientes que deben ser enteros.

In [4]:
df_bank_copy = bank_df.copy()

In [5]:
# Aplicamos lambda para convertir a int, cuidando los NaN
df_bank_copy['age'] = df_bank_copy['age'].apply(lambda x: int(x) if pd.notnull(x) else pd.NA)

# Convertimos a tipo entero de pandas (Int64) para mantener los NaN
df_bank_copy['age'] = df_bank_copy['age'].astype('Int64')

# Comprobamos
print("-> age dtype:", df_bank_copy['age'].dtype)


-> age dtype: Int64


Vamos a normalizar las tres columnas booleanas (default, housing, loan) usando map.
Haremos el reemplazo 0 → "no" y 1 → "yes", para tener la misma nomenclatura que la columna y.

In [6]:
# Normalizamos columnas booleanas con map (0 -> 'no', 1 -> 'yes')

bool_cols = ['default', 'housing', 'loan']

for col in bool_cols:
    df_bank_copy[col] = df_bank_copy[col].map({0: 'no', 1: 'yes'})
    print(f"-> {col}: valores únicos tras normalización:", df_bank_copy[col].unique())
    # mostramos los valores únicos de cada columna para comprobar que solo 
    # quedan "no" y "yes" (y NaN si existieran).


-> default: valores únicos tras normalización: ['no' nan 'yes']
-> housing: valores únicos tras normalización: ['no' 'yes' nan]
-> loan: valores únicos tras normalización: ['no' 'yes' nan]


In [7]:
df_bank_copy[bool_cols].sample(5)

,default,housing,loan
40936,no,no,no
10333,no,yes,no
27083,no,no,no
1913,NaN,no,no
19828,no,no,no


Vamos a convertir las columnas:
- 'cons.price.idx'
- 'cons.conf.idx'
- 'euribor3m'
- 'nr.employed'

De tipo object -> float

aplicamos .str.replace(',', '.') para pasar las comas decimales a puntos.

In [8]:
# Convertimos columnas 'object' con comas decimales a float

conv_float = ['cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']

for col in conv_float:
    if df_bank_copy[col].dtype == 'object': #nos aseguramos que es tipo obj
        # Reemplazar comas por puntos y convertir a float
        df_bank_copy[col] = df_bank_copy[col].str.replace(',', '.', regex=False).astype(float)
        print(f"-> {col} convertido a float")
    else:
        print(f"-> {col} ya es {df_bank_copy[col].dtype}, no necesita conversión")
    
#Comprobamos que ahora son tipo float
display(df_bank_copy[conv_float].sample(5))
df_bank_copy[conv_float].dtypes


-> cons.price.idx convertido a float
-> cons.conf.idx convertido a float
-> euribor3m convertido a float
-> nr.employed convertido a float


,cons.price.idx,cons.conf.idx,euribor3m,nr.employed
28802,93.075,-47.1,1.410,5099.1
7738,93.994,-36.4,NaN,5191.0
11992,94.465,-41.8,4.958,5228.1
12379,93.918,-42.7,4.960,5228.1
14902,93.918,-42.7,4.957,5228.1


cons.price.idx    float64
cons.conf.idx     float64
euribor3m         float64
nr.employed       float64
dtype: object

Reconsideramos que la columna nr.employed debería ser int ya que no tiene mucho sentido tener un número decimal de empleados. 

In [9]:
# Conversión de nr.employed a entero
    
# Truncamos a int (manteniendo NaN donde corresponda)
df_bank_copy['nr.employed'] = df_bank_copy['nr.employed'].apply(lambda x: int(x) if pd.notnull(x) else pd.NA)
    
# Convertimos a tipo entero de pandas (Int64) para mantener los NaN
df_bank_copy['nr.employed'] = df_bank_copy['nr.employed'].astype('Int64')
    
print("-> 'nr.employed' convertido a entero Int64")
df_bank_copy['nr.employed'].sample(5)


-> 'nr.employed' convertido a entero Int64


30133    5099
24294    5195
6794     5191
39885    4991
15145    5228
Name: nr.employed, dtype: Int64

Convertimos columna date de object a datetime

In [10]:
df_bank_copy['date'].sample(5)

22915         1-junio-2016
2030          1-enero-2017
4028       28-febrero-2015
388      25-diciembre-2019
6238       17-octubre-2016
Name: date, dtype: object

Mapeamos manualmente los meses en español y los convertimos a números

In [11]:
meses = {
    "enero":"01","febrero":"02","marzo":"03","abril":"04",
    "mayo":"05","junio":"06","julio":"07","agosto":"08",
    "septiembre":"09","octubre":"10","noviembre":"11","diciembre":"12"
}

#Mapeamos de forma manual, cambiamos meses en letra por su correspondiente número de mes
for mes, num in meses.items():
    df_bank_copy['date'] = df_bank_copy['date'].str.replace(mes, num, regex=True)

#Convertimos a datetime
df_bank_copy['date'] = pd.to_datetime(df_bank_copy['date'], dayfirst=True)

#Hacemos un sample para comprobar
df_bank_copy['date'].sample(5)

4125    2016-05-12
37662   2015-11-14
27838   2019-11-11
27776   2018-03-13
7308    2019-07-20
Name: date, dtype: datetime64[ns]

Creamos las columnas contact_month y contact_year a partir de la columna date

In [12]:
df_bank_copy["contact_month"] = df_bank_copy["date"].dt.month_name()
df_bank_copy["contact_year"] = df_bank_copy["date"].dt.year

# Conversión contact_year en int
# Truncamos a int (manteniendo NaN donde corresponda)
df_bank_copy['contact_year'] = df_bank_copy['contact_year'].apply(lambda x: int(x) if pd.notnull(x) else pd.NA)
    
# Convertimos a tipo entero de pandas (Int64) para mantener los NaN
df_bank_copy['contact_year'] = df_bank_copy['contact_year'].astype('Int64')


# Verificamos el resultado en algunas filas
df_bank_copy[["date", "contact_month", "contact_year"]].sample(5)


,date,contact_month,contact_year
25416,2019-07-18,July,2019
9408,2016-06-12,June,2016
16236,2016-05-22,May,2016
19683,2017-10-04,October,2017
38310,2018-06-19,June,2018


In [13]:
df_bank_copy.dtypes

age                        Int64
job                       object
marital                   object
education                 object
default                   object
housing                   object
loan                      object
contact                   object
duration                   int64
campaign                   int64
pdays                      int64
previous                   int64
poutcome                  object
emp.var.rate             float64
cons.price.idx           float64
cons.conf.idx            float64
euribor3m                float64
nr.employed                Int64
y                         object
date              datetime64[ns]
latitude                 float64
longitude                float64
id_                       object
contact_month             object
contact_year               Int64
dtype: object

Guardamos en la carpeta de DataProcessed el archivo con el dataframe limpio y con las transformaciones

In [14]:
df_bank_copy.to_csv("../Data/DataProcessed/bank-additional-clean.csv", index=False)
# Utilizamos index=False para evitar que se guarde la columna del índice como una columna extra en el CSV.